# 02 · Preprocesamiento y versión

De las fotos crudas a una versión usable del dataset. Cada etapa deja su
resultado en disco y salta lo ya hecho, así que se pueden correr en días
distintos: detectar hoy, recortar mañana.

| etapa | deja en disco |
|---|---|
| detección | `cache/detecciones.csv` |
| métricas | `cache/metricas.csv` |
| insecto visible | `cache/clip.csv` |
| auditoría | `auditoria.csv` |
| recortes | `procesado/<clase>/` |
| reparto | `manifiesto.json` |

El manifiesto se escribe al final: abandonar a mitad no deja una versión a
medias en el registro.

In [1]:
import json
from datetime import datetime, timezone
import time
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd

from corpus import (AUDITORIA, CACHE, IMAGENES, LISTA_ESPECIES, MANIFIESTO, METADATA, PESOS,
                    PROCESADO, escribirCsv, leerCsv)
SEMILLA = 42
NOTAS = "Dataset de insectos plaga del Tolima, con especies y taxon_id tomados de iNaturalist"

UMBRALES = {"nitidez": 60.0, "fracSaturados": 0.12, "fracNegros": 0.20, "ladoOriginal": 200}
MINIMO_POR_ESPECIE = 40

LADO_METRICA = 384
LADO_SALIDA = 448
CALIDAD_JPEG = 92
UMBRAL_DUP = 8
UMBRAL_RAFAGA = 12

DETECCIONES = CACHE / "detecciones.csv"
METRICAS = CACHE / "metricas.csv"
SPLITS = ("train", "test", "validate")
PROPORCIONES = (0.70, 0.20)

# species_list.json manda: una carpeta en Insectos/ o una fila en caché de una especie
# retirada no entra al corpus
ESPECIES = {s["name"].lower().replace(" ", "_")
            for s in json.loads(LISTA_ESPECIES.read_text(encoding="utf-8"))["species"]}

# especies que no se distinguen por foto comparten clase; la auditoría guarda la especie real
FUSIONES = {"atta_cephalotes": "atta_spp", "atta_colombica": "atta_spp"}

print(f"{NOTAS}\n{len(ESPECIES)} especies")

Dataset de insectos plaga del Tolima, con especies y taxon_id tomados de iNaturalist
36 especies


## 1 · Detección

YOLO-World localiza el insecto. La cadena vacía en los prompts es un sumidero
de fondo: sin ella el modelo deja sin caja 10 de cada 40 fotos, con ella 1 de 40,
y nunca gana.

In [2]:
PROMPTS = ("insect", "caterpillar", "beetle", "")
CLASES_VALIDAS = 3
CONFIANZA_MIN = 0.05
IMGSZ = 640
MARGEN = 0.15
AREA_MIN_RELATIVA = 0.005
COLUMNAS_DET = ["photo_id", "especie", "archivo", "ancho", "alto",
                "x1", "y1", "x2", "y2", "confianza"]


def cargarModelo():
    from ultralytics import YOLOWorld
    from ultralytics.nn import text_model

    PESOS.mkdir(exist_ok=True)
    text_model.WEIGHTS_DIR = PESOS          # si no, deja una segunda carpeta de pesos en la raíz
    pesos = PESOS / "yolov8s-worldv2.pt"
    modelo = YOLOWorld(str(pesos) if pesos.exists() else pesos.name)
    modelo.set_classes(list(PROMPTS))
    return modelo


def mejorCaja(resultado):
    cajas = resultado.boxes
    alto, ancho = resultado.orig_shape
    mejor, mejorConfianza = None, -1.0
    for i in range(len(cajas)):
        if int(cajas.cls[i]) >= CLASES_VALIDAS:
            continue
        x1, y1, x2, y2 = (float(v) for v in cajas.xyxy[i])
        if (x2 - x1) * (y2 - y1) < AREA_MIN_RELATIVA * ancho * alto:
            continue
        if float(cajas.conf[i]) > mejorConfianza:
            mejor, mejorConfianza = (x1, y1, x2, y2), float(cajas.conf[i])
    return mejor, mejorConfianza


def photoIdDe(ruta):
    return ruta.stem.rsplit("_", 1)[-1]

In [3]:
rutas = [p for carpeta in sorted(IMAGENES.iterdir())
         if carpeta.is_dir() and carpeta.name in ESPECIES
         for p in sorted(carpeta.glob("*.jpg"))]
detecciones = leerCsv(DETECCIONES)
pendientes = [r for r in rutas if photoIdDe(r) not in {d["photo_id"] for d in detecciones}]
print(f"{len(detecciones)} en caché · {len(pendientes)} pendientes")

if pendientes:
    modelo, tanda, inicio = cargarModelo(), 64, time.time()
    for comienzo in range(0, len(pendientes), tanda):
        lote = pendientes[comienzo:comienzo + tanda]
        resultados = modelo.predict([str(r) for r in lote], imgsz=IMGSZ,
                                    conf=CONFIANZA_MIN, verbose=False)
        for ruta, resultado in zip(lote, resultados):
            alto, ancho = resultado.orig_shape
            caja, confianza = mejorCaja(resultado)
            x1, y1, x2, y2 = caja if caja else ("", "", "", "")
            detecciones.append({
                "photo_id": photoIdDe(ruta), "especie": ruta.parent.name,
                "archivo": ruta.name, "ancho": ancho, "alto": alto,
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                "confianza": confianza if caja else "",
            })
        escribirCsv(DETECCIONES, detecciones, COLUMNAS_DET)   # guardar por tanda: interrumpir no pierde trabajo
        hechas = min(comienzo + tanda, len(pendientes))
        ritmo = (time.time() - inicio) / hechas
        print(f"  {hechas}/{len(pendientes)} · {ritmo:.2f} s/img · "
              f"faltan {ritmo * (len(pendientes) - hechas) / 60:.0f} min")
# el caché sobrevive a las versiones: se conserva entero en disco, pero esta
# corrida solo describe las imágenes que existen ahora
detecciones = [d for d in detecciones if d["especie"] in ESPECIES]
print(f"{len(detecciones)} detecciones")

20227 en caché · 0 pendientes
19677 detecciones


## 2 · Métricas de calidad

El cuadrado se desplaza ante un borde en vez de rellenarse: un borde negro o
reflejado es una textura sintética que la red aprende como atajo.

Se miden a 384×384 para que el umbral de nitidez no dependa del tamaño original.

In [4]:
COLUMNAS_MET = ["photo_id", "cuadradoX", "cuadradoY", "ladoOriginal", "nitidez",
                "brillo", "fracSaturados", "fracNegros", "phash", "dhash"]
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def redimensionar(imagen, lado):
    interpolacion = cv2.INTER_AREA if imagen.shape[0] > lado else cv2.INTER_CUBIC
    return cv2.resize(imagen, (lado, lado), interpolation=interpolacion)


def recortar(imagen, x, y, lado):
    return imagen[y:y + lado, x:x + lado]


def cuadradoDe(deteccion):
    if deteccion["x1"] == "":
        return None
    x1, y1, x2, y2 = (float(deteccion[k]) for k in ("x1", "y1", "x2", "y2"))
    ancho, alto = int(deteccion["ancho"]), int(deteccion["alto"])
    lado = min(max(x2 - x1, y2 - y1) * (1 + 2 * MARGEN), ancho, alto)
    x = min(max((x1 + x2) / 2 - lado / 2, 0), ancho - lado)
    y = min(max((y1 + y2) / 2 - lado / 2, 0), alto - lado)
    return int(round(x)), int(round(y)), int(round(lado))


def empaquetar(bits):
    valor = 0
    for bit in bits:
        valor = (valor << 1) | int(bit)
    return f"{valor:016x}"


def metricasDe(recorte):
    normalizado = redimensionar(recorte, LADO_METRICA)
    gris = cv2.cvtColor(normalizado, cv2.COLOR_BGR2GRAY)
    valor = cv2.cvtColor(normalizado, cv2.COLOR_BGR2HSV)[:, :, 2]

    dct = cv2.dct(np.float32(cv2.resize(gris, (32, 32), interpolation=cv2.INTER_AREA)))[:8, :8]
    chica = cv2.resize(gris, (9, 8), interpolation=cv2.INTER_AREA)
    return {
        "nitidez": float(cv2.Laplacian(gris, cv2.CV_64F).var()),
        "brillo": float(np.median(valor)),
        "fracSaturados": float(np.mean(valor >= 250)),
        "fracNegros": float(np.mean(valor <= 5)),
        "phash": empaquetar((dct > np.median(dct)).ravel()),
        "dhash": empaquetar((chica[:, 1:] > chica[:, :-1]).ravel()),
    }

In [5]:
metricas = leerCsv(METRICAS)
conocidas = {m["photo_id"] for m in metricas}
pendientes = [d for d in detecciones if d["photo_id"] not in conocidas]
print(f"{len(metricas)} en caché · {len(pendientes)} pendientes")

for numero, deteccion in enumerate(pendientes, start=1):
    cuadrado = cuadradoDe(deteccion)
    if cuadrado is None:
        continue
    x, y, lado = cuadrado
    imagen = cv2.imread(str(IMAGENES / deteccion["especie"] / deteccion["archivo"]))
    if imagen is None or lado < 8:
        continue
    fila = {"photo_id": deteccion["photo_id"], "cuadradoX": x, "cuadradoY": y,
            "ladoOriginal": lado}
    fila.update(metricasDe(recortar(imagen, x, y, lado)))
    metricas.append(fila)
    if numero % 500 == 0:
        escribirCsv(METRICAS, metricas, COLUMNAS_MET)
        print(f"  {numero}/{len(pendientes)}")

escribirCsv(METRICAS, metricas, COLUMNAS_MET)
print(f"{len(metricas)} métricas")

17348 en caché · 2675 pendientes
17348 métricas


## 3 · Insecto visible

En hormigas, YOLO-World suele encuadrar el hormiguero, el sendero o la hojarasca, y
esas fotos son nítidas, así que pasan los filtros de calidad. CLIP compara el recorte
con "hormiga" frente a "hormiguero, suelo, hojarasca, planta, flor, mano" y descarta
lo que queda bajo `UMBRAL_CLIP`. Solo se aplica a `FILTRO_CLIP`: en otras clases
(escarabajos, mariposas) el texto "hormiga" no tiene sentido.

Si cambian los textos, borrar `cache/clip.csv`; el umbral se aplica después y no lo exige.

In [6]:
FILTRO_CLIP = {"atta_cephalotes", "atta_colombica", "solenopsis_geminata"}
TEXTOS_CLIP = ("a close-up photo of an ant", "a photo of ants on a leaf",
               "a photo of an anthill", "a photo of soil and dirt",
               "a photo of leaf litter on the ground", "a photo of a plant",
               "a photo of a flower", "a photo of a hand")
POSITIVOS_CLIP = 2      # los primeros textos de TEXTOS_CLIP
UMBRAL_CLIP = 0.30
CLIP_CSV = CACHE / "clip.csv"


def cargarClip():
    import clip
    import torch

    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    modelo, preparar = clip.load(str(PESOS / "clip" / "ViT-B-32.pt"), device=dispositivo)
    with torch.no_grad():
        textos = modelo.encode_text(clip.tokenize(list(TEXTOS_CLIP)).to(dispositivo))
    return modelo, preparar, textos / textos.norm(dim=-1, keepdim=True), dispositivo


def puntajesClip(cargado, recortes):
    """Probabilidad de los textos positivos frente al resto, por recorte."""
    import torch
    from PIL import Image

    modelo, preparar, textos, dispositivo = cargado
    lote = torch.stack([preparar(Image.fromarray(cv2.cvtColor(r, cv2.COLOR_BGR2RGB)))
                        for r in recortes]).to(dispositivo)
    with torch.no_grad():
        imagenes = modelo.encode_image(lote)
        imagenes /= imagenes.norm(dim=-1, keepdim=True)
        probabilidades = (100 * imagenes @ textos.T).softmax(-1)
    return probabilidades[:, :POSITIVOS_CLIP].sum(-1).tolist()

In [7]:
filasClip = leerCsv(CLIP_CSV)
conocidas = {f["photo_id"] for f in filasClip}
cuadrados = {m["photo_id"]: m for m in metricas}
candidatas = [d for d in detecciones
              if d["especie"] in FILTRO_CLIP and d["photo_id"] in cuadrados]
pendientes = [d for d in candidatas if d["photo_id"] not in conocidas]
print(f"{len(filasClip)} en caché · {len(pendientes)} pendientes")

if pendientes:
    cargado, tanda = cargarClip(), 64
    for comienzo in range(0, len(pendientes), tanda):
        lote = pendientes[comienzo:comienzo + tanda]
        recortes = []
        for deteccion in lote:
            m = cuadrados[deteccion["photo_id"]]
            imagen = cv2.imread(str(IMAGENES / deteccion["especie"] / deteccion["archivo"]))
            recortes.append(recortar(imagen, int(m["cuadradoX"]), int(m["cuadradoY"]),
                                     int(m["ladoOriginal"])))
        for deteccion, puntaje in zip(lote, puntajesClip(cargado, recortes)):
            filasClip.append({"photo_id": deteccion["photo_id"], "puntajeClip": puntaje})
        escribirCsv(CLIP_CSV, filasClip, ["photo_id", "puntajeClip"])

vigentes = {d["photo_id"] for d in candidatas}
puntajeClip = {f["photo_id"]: float(f["puntajeClip"]) for f in filasClip
               if f["photo_id"] in vigentes}
bajo = sum(p < UMBRAL_CLIP for p in puntajeClip.values())
print(f"{len(puntajeClip)} puntuadas · {bajo} bajo {UMBRAL_CLIP}")

1103 en caché · 0 pendientes
1103 puntuadas · 160 bajo 0.3


## 4 · Auditoría

Cascada excluyente: el primer motivo que aplica gana, así el resumen suma el
corpus entero sin doble contabilidad. La deduplicación va al final para que el
representante conservado ya haya pasado los filtros de calidad.

Los umbrales están congelados como literales a propósito: recalcularlos por
percentil cambiaría el dataset cada vez que el corpus crece.

In [8]:
MOTIVOS = ("sin_deteccion", "recorte_pequeno", "desenfocada", "sobreexpuesta",
           "subexpuesta", "sin_insecto_visible", "casi_duplicado", "especie_escasa", "aceptada")
COLUMNAS_AUD = ["especie", "photo_id", "observation_id", "archivo", "ancho", "alto",
                "x1", "y1", "x2", "y2", "confianza", "cuadradoX", "cuadradoY",
                "ladoOriginal", "nitidez", "brillo", "fracSaturados", "fracNegros",
                "phash", "dhash", "puntajeClip", "grupo", "veredicto", "motivo", "split"]


def motivoDeCalidad(fila):
    if float(fila["ladoOriginal"]) < UMBRALES["ladoOriginal"]:
        return "recorte_pequeno"
    if float(fila["nitidez"]) < UMBRALES["nitidez"]:
        return "desenfocada"
    if float(fila["fracSaturados"]) > UMBRALES["fracSaturados"]:
        return "sobreexpuesta"
    if float(fila["fracNegros"]) > UMBRALES["fracNegros"]:
        return "subexpuesta"
    return None


def distanciasEn(hashes):
    matriz = np.array([[(int(h, 16) >> (8 * i)) & 0xFF for i in range(8)]
                       for h in hashes], dtype=np.uint8)
    return POPCOUNT[matriz[:, None, :] ^ matriz[None, :, :]].sum(axis=2)


def paresCercanos(filas):
    """Compartir observación relaja el umbral, pero vistas distintas del mismo animal distan más y sobreviven."""
    porEspecie = defaultdict(list)
    for fila in filas:
        porEspecie[fila["especie"]].append(fila)

    pares = []
    for grupo in porEspecie.values():
        if len(grupo) < 2:
            continue
        for nombre in ("phash", "dhash"):
            distancias = distanciasEn([f[nombre] for f in grupo])
            for i, j in zip(*np.triu_indices(len(grupo), k=1)):
                misma = grupo[i]["observation_id"] == grupo[j]["observation_id"]
                if distancias[i, j] <= (UMBRAL_RAFAGA if misma else UMBRAL_DUP):
                    pares.append((grupo[i]["photo_id"], grupo[j]["photo_id"]))
    return pares


def agrupar(ids, pares):
    padre = {i: i for i in ids}

    def raiz(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    for a, b in pares:
        if raiz(a) != raiz(b):
            padre[raiz(b)] = raiz(a)

    grupos = defaultdict(list)
    for i in ids:
        grupos[raiz(i)].append(i)
    return [sorted(g) for g in grupos.values() if len(g) > 1]

In [9]:
porId = {m["photo_id"]: m for m in metricas}
observacionDe = {m["photo_id"]: m["observation_id"] for m in leerCsv(METADATA)}

auditoria = []
for deteccion in detecciones:
    metrica = porId.get(deteccion["photo_id"])
    fila = dict.fromkeys(COLUMNAS_AUD, "")
    fila.update(deteccion)
    fila["observation_id"] = observacionDe.get(deteccion["photo_id"], "")
    fila.update(metrica or {})
    fila["motivo"] = "sin_deteccion" if metrica is None else motivoDeCalidad(metrica)
    if deteccion["photo_id"] in puntajeClip:
        fila["puntajeClip"] = puntajeClip[deteccion["photo_id"]]
        if not fila["motivo"] and puntajeClip[deteccion["photo_id"]] < UMBRAL_CLIP:
            fila["motivo"] = "sin_insecto_visible"
    auditoria.append(fila)

vivas = [f for f in auditoria if not f["motivo"]]
if vivas:
    porFoto = {f["photo_id"]: f for f in vivas}
    nitidez = {i: float(f["nitidez"]) for i, f in porFoto.items()}
    for numero, grupo in enumerate(agrupar(list(porFoto), paresCercanos(vivas)), start=1):
        elegido = min(grupo, key=lambda i: (-nitidez[i], int(i)))
        for photoId in grupo:
            porFoto[photoId]["grupo"] = numero
            if photoId != elegido:
                porFoto[photoId]["motivo"] = "casi_duplicado"

conteo = defaultdict(int)
for fila in auditoria:
    if not fila["motivo"]:
        conteo[fila["especie"]] += 1
escasas = {e for e, n in conteo.items() if n < MINIMO_POR_ESPECIE}
for fila in auditoria:
    if not fila["motivo"] and fila["especie"] in escasas:
        fila["motivo"] = "especie_escasa"

for fila in auditoria:
    fila["motivo"] = fila["motivo"] or "aceptada"
    fila["veredicto"] = "aceptada" if fila["motivo"] == "aceptada" else "descartada"

resumen = defaultdict(int)
for fila in auditoria:
    resumen[fila["motivo"]] += 1
pd.DataFrame([{"motivo": m, "imagenes": resumen[m],
               "porcentaje": round(100 * resumen[m] / len(auditoria), 2)}
              for m in MOTIVOS if resumen[m]])

,motivo,imagenes,porcentaje
0,sin_deteccion,2675,13.59
1,recorte_pequeno,713,3.62
2,desenfocada,936,4.76
3,sobreexpuesta,782,3.97
4,subexpuesta,142,0.72
5,sin_insecto_visible,137,0.70
6,casi_duplicado,667,3.39
7,aceptada,13625,69.24


## 5 · Recortes

Salta los que ya existen, así que reejecutar es barato.

In [10]:
PROCESADO.mkdir(exist_ok=True)
escritas = omitidas = 0
ilegibles = []
for fila in auditoria:
    if fila["veredicto"] != "aceptada":
        continue
    carpeta = PROCESADO / FUSIONES.get(fila["especie"], fila["especie"])
    carpeta.mkdir(parents=True, exist_ok=True)
    salida = carpeta / fila["archivo"]
    if salida.exists():
        omitidas += 1
        continue
    imagen = cv2.imread(str(IMAGENES / fila["especie"] / fila["archivo"]))
    if imagen is None:
        ilegibles.append(f'{fila["especie"]}/{fila["archivo"]}')
        continue
    recorte = recortar(imagen, int(fila["cuadradoX"]), int(fila["cuadradoY"]),
                       int(fila["ladoOriginal"]))
    cv2.imwrite(str(salida), redimensionar(recorte, LADO_SALIDA),
                [cv2.IMWRITE_JPEG_QUALITY, CALIDAD_JPEG])
    escritas += 1

escribirCsv(AUDITORIA, auditoria, COLUMNAS_AUD)
print(f"escritas {escritas} · ya estaban {omitidas}")

# el reparto sale de la auditoría, no de los archivos: una fuente ilegible deja
# el manifiesto apuntando a recortes que nunca se escribieron
if ilegibles:
    raise RuntimeError(f"{len(ilegibles)} fuentes ilegibles: {ilegibles[:5]}")

escritas 1 · ya estaban 13624


## 6 · Reparto y publicación

La semilla se combina con el nombre de la especie, de modo que agregar una
especie nueva no altera el reparto de las que ya estaban.

In [11]:
import random

aceptadas = defaultdict(lambda: defaultdict(list))
for fila in auditoria:
    if fila["veredicto"] == "aceptada":
        aceptadas[FUSIONES.get(fila["especie"], fila["especie"])][fila["observation_id"]].append(fila["archivo"])

# observaciones enteras a un solo split: fotos del mismo individuo no cruzan a evaluación
reparto = {}
for clase in sorted(aceptadas):
    porObservacion = aceptadas[clase]
    total = sum(len(a) for a in porObservacion.values())
    cortes = (round(total * PROPORCIONES[0]), round(total * sum(PROPORCIONES)))
    ids = random.Random(f"{SEMILLA}:{clase}").sample(sorted(porObservacion), len(porObservacion))
    reparto[clase] = {s: [] for s in SPLITS}
    asignadas = 0
    for observacion in ids:
        split = SPLITS[0] if asignadas < cortes[0] else SPLITS[1] if asignadas < cortes[1] else SPLITS[2]
        reparto[clase][split].extend(sorted(porObservacion[observacion]))
        asignadas += len(porObservacion[observacion])

splitDe = {a: s for r in reparto.values() for s in SPLITS for a in r[s]}
for fila in auditoria:
    fila["split"] = splitDe.get(fila["archivo"], "")
escribirCsv(AUDITORIA, auditoria, COLUMNAS_AUD)

conteos = {s: sum(len(r[s]) for r in reparto.values()) for s in SPLITS}
conteos["total"] = sum(conteos.values())

MANIFIESTO.write_text(json.dumps({
    "creado": datetime.now(timezone.utc).isoformat(),
    "notas": NOTAS,
    "parametros": {"umbrales": UMBRALES, "minimo_por_especie": MINIMO_POR_ESPECIE,
                   "semilla": SEMILLA, "fusiones": FUSIONES, "lado_salida": LADO_SALIDA},
    "num_clases": len(reparto),
    "conteos": conteos,
    "reparto": reparto,
}, indent=1, ensure_ascii=False), encoding="utf-8")

print(f"{len(reparto)} clases · {conteos}")

35 clases · {'train': 9554, 'test': 2724, 'validate': 1347, 'total': 13625}


## Fuga entre splits

Porcentaje del test cuya observación también aparece en train: son fotos del
mismo individuo, así que inflan la métrica.

In [12]:
observacionPorArchivo = {m["archivo"]: m["observation_id"] for m in leerCsv(METADATA)}
enTrain = {observacionPorArchivo[a] for clase in reparto for a in reparto[clase]["train"]}
test = [a for clase in reparto for a in reparto[clase]["test"]]
fugadas = sum(1 for a in test if observacionPorArchivo[a] in enTrain)
print(f"{fugadas}/{len(test)} ({fugadas / len(test):.1%})")

0/2724 (0.0%)


Para usarla, en el `corpus.json` del repositorio de modelado poner `"version"` con este
nombre.

In [13]:
pd.Series({**json.loads(MANIFIESTO.read_text())["conteos"],
           "clases": json.loads(MANIFIESTO.read_text())["num_clases"]}).to_frame("valor")

,valor
train,9554
test,2724
validate,1347
total,13625
clases,35
